# Dynamics 06 — Preuves bornées et coût du raisonnement (Math for AI Safety)

**Série** : GameTheory / Dynamics — jumeau Rust du carnet Python
**Source du portage** : [Dynamics-06-Preuves-Bornees-Cout-du-Raisonnement-Python.ipynb](Dynamics-06-Preuves-Bornees-Cout-du-Raisonnement-Python.ipynb)
**Kernel** : `rust` (evcxr, `std` seul) — mêmes témoins recalculés : seuil DUPOC(k), coût ε × profondeur, contrôle négatif

**Navigation** : [<< Dynamics-05-Agents-Bornes-Python](Dynamics-05-Agents-Bornes-Python.ipynb) | [Index](README.md) | [Dynamics-07-Equilibres-par-Simulation-Python >>](Dynamics-07-Equilibres-par-Simulation-Python.ipynb) | [Dynamics-06 Python — source du portage](Dynamics-06-Preuves-Bornees-Cout-du-Raisonnement-Python.ipynb)

Prolongement formel du chantier : [→ GameTheory-06k — FairBot par Löb](Dynamics-04b-FairBot-Loeb-Lean.ipynb), qui franchit en Lean la limite que ce carnet laisse ouverte.

Ce notebook prolonge `Dynamics-04-Open-Source-Game-Theory-Python.ipynb`. Le moteur de 06e — bots sous forme de programmes (_toy), matrice PD canonique (T=5, R=3, P=1, S=0, condition stricte `2R > T + S`), bornes `MAX_DEPTH` et `STEP_BUDGET`, exception `SimulationTimeout` — est **reproduit** ici avec un instrument **identique**. L objet de 06f est ce que 06e n instrumente pas : **la borne elle-meme**, sa variation, et le cout qu elle impose au raisonnement.

## Vocabulaire honnete — non negociable

Ce que mesure ce notebook est un **comportement observe sur une famille finie sous une borne donnee**. Le theoreme parametrique borne de Lob (Critch 2016, *A priori refinement*, arXiv 1602.04184) se prouve autrement, en logique modale constructive, et le resultat est d un autre registre. Toute phrase qui laisserait entendre que la mesure **etablit** ou **demontre** le theoreme est a reecrire en « coherent avec ». Le corpus de reference est archive sous `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\` (Critch 2016 ; Garrabrant 2018).

## Plan du notebook

1. **Instrument IDENTIQUE des deux cotes** de toute comparaison — sinon on mesure son instrument, pas l objet.
2. **Seuil `DUPOC(k)` self-play AVEC LA COURBE** — faire varier `k` et montrer la valeur du basculement, pas juste la valeur.
3. **Cout `ε × profondeur`** — reconstruire les equilibres purs/mixte de CooperateBot / FairBot / PrudentBot sous ce cout.
4. **Controle negatif obligatoire** — budget trop faible OU cout trop eleve **restaure la defection**. Sans lui, rien ne prouve que la borne agit .
5. **≥ 3 exercices** non resolus, a stubs executables (`pass` / `print` / `return None`) — le notebook reste executable de bout en bout, exercices non faits.

## Exigence de non-trivialite

La borne doit **changer un resultat** : il faut au moins un couple de bots dont l issue differe selon `k` ou selon le cout. Un balayage de `k` ou rien ne bascule reclame un probleme plus riche, pas un commentaire d excuse.

## Perimetre

`GameTheory-06e` est **hors perimetre**. Le notebook 06f est **autonome** : moteur reproduit localement (instrument identique = memes constantes, memes bornes, meme exception), pas d import cross-fichier.

## 0. Preliminaires — moteur reproduit (instrument IDENTIQUE 06e)

Constantes PD canoniques, condition stricte, bornes et exception `SimulationTimeout`. Toute comparaison ulterieure utilise **exactement** ces valeurs, sans quoi le point 1 de l acceptance est violé.

In [2]:
const T: i64 = 5; const R: i64 = 3; const P: i64 = 1; const S: i64 = 0;
const MAX_DEPTH: i64 = 3;
const STEP_BUDGET: i64 = 1000;

// Constantes PD canoniques (T=5, R=3, P=1, S=0) et bornes du moteur 
// (MAX_DEPTH, STEP_BUDGET) -- instrument IDENTIQUE au 06e ; la garde ci-dessous
// verifie la PD stricte : rang T > R > P > S et cooperation mutuelle > alternance.
assert!(T > R && R > P && P > S, "Parametres PD non canoniques");
assert!(2 * R > T + S, "Cooperation mutuelle > alternance (stricte PD)");

#[derive(Clone, Copy, PartialEq)]
enum Act { C, D }

impl Act {
    fn name(self) -> &'static str {
        match self {
            Act::C => "C",
            Act::D => "D",
        }
    }
}

fn payoff_self(my_action: Act, other_action: Act) -> (i64, i64) {
    match (my_action, other_action) {
        (Act::C, Act::C) => (R, R),
        (Act::C, Act::D) => (S, T),
        (Act::D, Act::C) => (T, S),
        (Act::D, Act::D) => (P, P),
    }
}

println!("PD canonique OK : T={}, R={}, P={}, S={}, 2R={} > T+S={}", T, R, P, S, 2 * R, T + S);
println!("Bornes : MAX_DEPTH={}, STEP_BUDGET={}", MAX_DEPTH, STEP_BUDGET);
println!("Exception : SimulationTimeout levee avant verdict si budget epuise");

PD canonique OK : T=5, R=3, P=1, S=0, 2R=6 > T+S=5


Bornes : MAX_DEPTH=3, STEP_BUDGET=1000


Exception : SimulationTimeout levee avant verdict si budget epuise


La sortie confirme l'installation des constantes PD canoniques : `T=5, R=3, P=1, S=0` avec la condition stricte `2R=6 > T+S=5`, ainsi que les bornes `MAX_DEPTH=3` et `STEP_BUDGET=1000`. L'exception `SimulationTimeout` est en place : elle se declenchera si le budget est epuise avant le verdict. Cette configuration identique a celle de 06e garantit que toute comparaison ulterieure mesure l'objet etudie (la borne et son cout) et non un artefact du moteur de simulation. Les trois lignes de sortie enumerent respectivement la validation PD, les bornes, et le contrat d'exception.

## 1. Bots — memes signatures que 06e (instrument IDENTIQUE)

Cinq bots : CooperateBot, DefectBot, FairBot (DUPOC simplifie : coopere si l autre source contient `return "C"`), CUPOD (Continuous Update of Prisoner s Dilemma), PrudentBot (defect sauf si preuve de cooperation stable). Vocabulaire **coherent avec** la litterature, pas de theorie forte engagee.

In [3]:
#[derive(Clone, Copy, PartialEq)]
enum Act { C, D }

impl Act {
    fn name(self) -> &'static str {
        match self {
            Act::C => "C",
            Act::D => "D",
        }
    }
}

enum Bot {
    Cooperate,
    Defect,
    Fair,
    Cupod { hist: Vec<&'static str> },
    Prudent,
}

impl Bot {
    fn name(&self) -> &'static str {
        match self {
            Bot::Cooperate => "CooperateBot_toy",
            Bot::Defect => "DefectBot_toy",
            Bot::Fair => "FairBot_toy",
            Bot::Cupod { .. } => "CUPOD_toy",
            Bot::Prudent => "PrudentBot_toy",
        }
    }

    fn act(&mut self, other_source: &str) -> Act {
        match self {
            Bot::Cooperate => Act::C,
            Bot::Defect => Act::D,
            Bot::Fair => if other_source.contains("return \"C\"") { Act::C } else { Act::D },
            Bot::Cupod { hist } => {
                // Continuous Update of PD : coopere tant que l autre a coopere au tour precedent
                if hist.is_empty() {
                    hist.push("C"); // premiere rencontre : ouverture cooperative
                    return Act::C;
                }
                if !other_source.is_empty() && other_source.contains("return \"D\"") && hist[hist.len() - 1] == "D" {
                    // defection repetee de l autre -> defection
                    hist.push("D");
                    return Act::D;
                }
                hist.push("C");
                Act::C
            }
            Bot::Prudent => {
                // Prudent : defaut D sauf si preuve de cooperation dans la source
                if other_source.contains("return \"C\"") && other_source.to_lowercase().contains("defect") {
                    // l autre a une strategie mixte documentee -> ouverture prudente
                    Act::C
                } else {
                    Act::D
                }
            }
        }
    }
}

fn py_float(x: f64) -> String {
    // repr Python d un flottant : les integraux s impriment avec un .0, les autres
    // en forme shortest round-trip (identique de Rust a CPython sur ce domaine).
    if x.is_finite() && x == x.trunc() { format!("{:.1}", x) } else { format!("{}", x) }
}

// Ordre d insertion de la source : les 5 noms _toy, puis leurs facteurs de cout.
const BOT_NAMES: [&str; 5] = ["CooperateBot_toy", "DefectBot_toy", "FairBot_toy", "CUPOD_toy", "PrudentBot_toy"];
const COST_FACTORS: [f64; 5] = [1.0, 1.0, 2.0, 1.5, 2.0];

let bots_list = format!("[{}]", BOT_NAMES.iter().map(|n| format!("'{}'", n)).collect::<Vec<_>>().join(", "));
let cost_dict = format!("{{{}}}", BOT_NAMES.iter().zip(COST_FACTORS.iter())
    .map(|(n, f)| format!("'{}': {}", n, py_float(*f)))
    .collect::<Vec<_>>().join(", "));

println!("Bots charges : {}", bots_list);
println!("Cost factors : {}", cost_dict);

Bots charges : ['CooperateBot_toy', 'DefectBot_toy', 'FairBot_toy', 'CUPOD_toy', 'PrudentBot_toy']


Cost factors : {'CooperateBot_toy': 1.0, 'DefectBot_toy': 1.0, 'FairBot_toy': 2.0, 'CUPOD_toy': 1.5, 'PrudentBot_toy': 2.0}


Cinq bots sont charges avec leurs facteurs de cout distincts : `CooperateBot_toy` et `DefectBot_toy` a `1.0` (cout minimal, branchements directs sans lecture de source), `FairBot_toy` et `PrudentBot_toy` a `2.0` (ils lisent et analysent la source de l'adversaire, ce qui coute plus cher en etats explores), et `CUPOD_toy` a `1.5` (memoire interne du dernier coup partagee entre les appels). La sortie affiche d'abord la liste des cinq bots, puis le dictionnaire des facteurs de cout. Ces facteurs rendront le cout du raisonnement mesurable par joueur dans la suite du notebook, notamment pour l'exercice 2 sur l'asymetrie des couts.

## 2. Moteur `simulate_payoff` — instrument de mesure (point 1)

Renvoie l issue d un duel entre deux bots sous la borne. Mesure : profondeur atteinte, nombre d etats explores, temps ecoule. **Meme instrument des deux cotes** de toute comparaison ulterieure.

In [4]:
const T: i64 = 5; const R: i64 = 3; const P: i64 = 1; const S: i64 = 0;
const MAX_DEPTH: i64 = 3;
const STEP_BUDGET: i64 = 1000;

#[derive(Clone, Copy, PartialEq)]
enum Act { C, D }

impl Act {
    fn name(self) -> &'static str {
        match self {
            Act::C => "C",
            Act::D => "D",
        }
    }
}

fn payoff_self(my_action: Act, other_action: Act) -> (i64, i64) {
    match (my_action, other_action) {
        (Act::C, Act::C) => (R, R),
        (Act::C, Act::D) => (S, T),
        (Act::D, Act::C) => (T, S),
        (Act::D, Act::D) => (P, P),
    }
}

enum Bot {
    Cooperate,
    Defect,
    Fair,
    Cupod { hist: Vec<&'static str> },
    Prudent,
}

impl Bot {
    fn name(&self) -> &'static str {
        match self {
            Bot::Cooperate => "CooperateBot_toy",
            Bot::Defect => "DefectBot_toy",
            Bot::Fair => "FairBot_toy",
            Bot::Cupod { .. } => "CUPOD_toy",
            Bot::Prudent => "PrudentBot_toy",
        }
    }

    fn act(&mut self, other_source: &str) -> Act {
        match self {
            Bot::Cooperate => Act::C,
            Bot::Defect => Act::D,
            Bot::Fair => if other_source.contains("return \"C\"") { Act::C } else { Act::D },
            Bot::Cupod { hist } => {
                // Continuous Update of PD : coopere tant que l autre a coopere au tour precedent
                if hist.is_empty() {
                    hist.push("C"); // premiere rencontre : ouverture cooperative
                    return Act::C;
                }
                if !other_source.is_empty() && other_source.contains("return \"D\"") && hist[hist.len() - 1] == "D" {
                    // defection repetee de l autre -> defection
                    hist.push("D");
                    return Act::D;
                }
                hist.push("C");
                Act::C
            }
            Bot::Prudent => {
                // Prudent : defaut D sauf si preuve de cooperation dans la source
                if other_source.contains("return \"C\"") && other_source.to_lowercase().contains("defect") {
                    // l autre a une strategie mixte documentee -> ouverture prudente
                    Act::C
                } else {
                    Act::D
                }
            }
        }
    }
}

fn _sources_of() -> Vec<(&'static str, &'static str)> {
    // Reconstruction manuelle des sources (string literal par bot).
    // Les bots _toy sont definis comme des fonctions ; leur "source" est le corps de la fonction.
    // On reproduit ce corps en string literal ici, instrument IDENTIQUE a 06e (pas d equivalent
    // Rust de inspect.getsource : le choix manuel preserve la stabilite byte-a-byte de la table).
    vec![
        ("CooperateBot_toy", "return \"C\""),
        ("DefectBot_toy", "return \"D\""),
        ("FairBot_toy", "if 'return \"C\"' in other_source: return \"C\"\nreturn \"D\""),
        ("CUPOD_toy", "if not _hist: _hist.append('C'); return 'C'\nreturn 'C'"),
        ("PrudentBot_toy", "if 'return \"C\"' in other_source and 'defect' in other_source.lower(): return \"C\"\nreturn \"D\""),
    ]
}

fn _src_of(sources: &[(&'static str, &'static str)], name: &str) -> &'static str {
    for (n, s) in sources {
        if *n == name {
            return s;
        }
    }
    ""
}

struct Metrics {
    states_explored: i64,
    elapsed_seconds: f64,
    depth_reached: i64,
}

fn metrics_dict(m: &Metrics) -> String {
    // Repr Python du dict metrics (ordre d insertion, apostrophes simples) ;
    // elapsed_seconds en notation scientifique Rust {:e} (grandeur murale, presence seule).
    format!("{{'states_explored': {}, 'elapsed_seconds': {:e}, 'depth_reached': {}}}",
            m.states_explored, m.elapsed_seconds, m.depth_reached)
}

fn simulate_payoff(bot_a: &mut Bot, bot_b: &mut Bot, sources: &[(&'static str, &'static str)],
                   max_depth: i64, step_cap: i64) -> (Act, Act, (i64, i64), &'static str, Metrics) {
    // Duel entre bot_a (moi) et bot_b (adversaire) sous la borne.
    // Retourne : (action_a, action_b, payoff, status, metrics)
    let src_a = _src_of(sources, bot_a.name());
    let src_b = _src_of(sources, bot_b.name());
    let mut states_explored: i64 = 0;
    let mut prev: Option<(Act, Act)> = None; // variable LOCALE : pas d etat partage entre runs
    let t0 = std::time::Instant::now();

    // Profondeur 1 - appel direct
    let mut a = bot_a.act(src_b);
    let mut b = bot_b.act(src_a);
    states_explored += 2;

    // Profondeur 2+ - iterations sous la borne (borne du range figee une fois)
    for _ in 0..std::cmp::min(max_depth - 1, step_cap - states_explored) {
        if states_explored >= step_cap {
            // SimulationTimeout : step_cap epuise AVANT calcul du verdict de l iteration
            let m = Metrics {
                states_explored,
                elapsed_seconds: t0.elapsed().as_secs_f64(),
                depth_reached: std::cmp::min(max_depth, states_explored / 2),
            };
            return (Act::D, Act::D, (P, P), "timeout", m);
        }
        a = bot_a.act(src_b);
        b = bot_b.act(src_a);
        states_explored += 2;
        if let Some((prev_a, _)) = prev {
            if a == b && b == prev_a {
                break; // point fixe atteint
            }
        }
        prev = Some((a, b));
    }

    let m = Metrics {
        states_explored,
        elapsed_seconds: t0.elapsed().as_secs_f64(),
        depth_reached: std::cmp::min(max_depth, states_explored / 2),
    };
    (a, b, payoff_self(a, b), "ok", m)
}

println!("Moteur simulate_payoff charge avec instrument (states_explored, elapsed_seconds, depth_reached).");

Moteur simulate_payoff charge avec instrument (states_explored, elapsed_seconds, depth_reached).


Le moteur `simulate_payoff` est charge avec son instrument de mesure : chaque appel retournera un tuple `(action_a, action_b, payoff, status, metrics)` ou `metrics` contient trois cles : `states_explored` (nombre d'etats visites), `elapsed_seconds` (temps ecoule en secondes) et `depth_reached` (profondeur atteinte dans l'arbre de simulation). La sortie confirme le chargement en une ligne. Cet instrument identique des deux cotes de toute comparaison est le garant du point 1 de l'acceptance : il permet de mesurer l'objet et non le moteur, condition necessaire pour attribuer toute difference observee a la variation d'un parametre.

## 3. Point 1 acceptance — instrument IDENTIQUE verifie

On execute `simulate_payoff` deux fois sur le meme duel. Les `metrics` doivent etre **bit-identiques** sur les metriques deterministes (memes etats et meme profondeur ; le temps mesure est une grandeur murale, non comparee). C est la garantie que les comparaisons ulterieures (point 2 `DUPOC(k)`, point 3 cout, point 4 controle negatif) mesurent bien l objet et pas le moteur.

In [5]:
const T: i64 = 5; const R: i64 = 3; const P: i64 = 1; const S: i64 = 0;
const MAX_DEPTH: i64 = 3;
const STEP_BUDGET: i64 = 1000;

#[derive(Clone, Copy, PartialEq)]
enum Act { C, D }

impl Act {
    fn name(self) -> &'static str {
        match self {
            Act::C => "C",
            Act::D => "D",
        }
    }
}

fn payoff_self(my_action: Act, other_action: Act) -> (i64, i64) {
    match (my_action, other_action) {
        (Act::C, Act::C) => (R, R),
        (Act::C, Act::D) => (S, T),
        (Act::D, Act::C) => (T, S),
        (Act::D, Act::D) => (P, P),
    }
}

enum Bot {
    Cooperate,
    Defect,
    Fair,
    Cupod { hist: Vec<&'static str> },
    Prudent,
}

impl Bot {
    fn name(&self) -> &'static str {
        match self {
            Bot::Cooperate => "CooperateBot_toy",
            Bot::Defect => "DefectBot_toy",
            Bot::Fair => "FairBot_toy",
            Bot::Cupod { .. } => "CUPOD_toy",
            Bot::Prudent => "PrudentBot_toy",
        }
    }

    fn act(&mut self, other_source: &str) -> Act {
        match self {
            Bot::Cooperate => Act::C,
            Bot::Defect => Act::D,
            Bot::Fair => if other_source.contains("return \"C\"") { Act::C } else { Act::D },
            Bot::Cupod { hist } => {
                // Continuous Update of PD : coopere tant que l autre a coopere au tour precedent
                if hist.is_empty() {
                    hist.push("C"); // premiere rencontre : ouverture cooperative
                    return Act::C;
                }
                if !other_source.is_empty() && other_source.contains("return \"D\"") && hist[hist.len() - 1] == "D" {
                    // defection repetee de l autre -> defection
                    hist.push("D");
                    return Act::D;
                }
                hist.push("C");
                Act::C
            }
            Bot::Prudent => {
                // Prudent : defaut D sauf si preuve de cooperation dans la source
                if other_source.contains("return \"C\"") && other_source.to_lowercase().contains("defect") {
                    // l autre a une strategie mixte documentee -> ouverture prudente
                    Act::C
                } else {
                    Act::D
                }
            }
        }
    }
}

fn _sources_of() -> Vec<(&'static str, &'static str)> {
    vec![
        ("CooperateBot_toy", "return \"C\""),
        ("DefectBot_toy", "return \"D\""),
        ("FairBot_toy", "if 'return \"C\"' in other_source: return \"C\"\nreturn \"D\""),
        ("CUPOD_toy", "if not _hist: _hist.append('C'); return 'C'\nreturn 'C'"),
        ("PrudentBot_toy", "if 'return \"C\"' in other_source and 'defect' in other_source.lower(): return \"C\"\nreturn \"D\""),
    ]
}

fn _src_of(sources: &[(&'static str, &'static str)], name: &str) -> &'static str {
    for (n, s) in sources {
        if *n == name {
            return s;
        }
    }
    ""
}

struct Metrics {
    states_explored: i64,
    elapsed_seconds: f64,
    depth_reached: i64,
}

fn metrics_dict(m: &Metrics) -> String {
    format!("{{'states_explored': {}, 'elapsed_seconds': {:e}, 'depth_reached': {}}}",
            m.states_explored, m.elapsed_seconds, m.depth_reached)
}

fn simulate_payoff(bot_a: &mut Bot, bot_b: &mut Bot, sources: &[(&'static str, &'static str)],
                   max_depth: i64, step_cap: i64) -> (Act, Act, (i64, i64), &'static str, Metrics) {
    let src_a = _src_of(sources, bot_a.name());
    let src_b = _src_of(sources, bot_b.name());
    let mut states_explored: i64 = 0;
    let mut prev: Option<(Act, Act)> = None;
    let t0 = std::time::Instant::now();

    let mut a = bot_a.act(src_b);
    let mut b = bot_b.act(src_a);
    states_explored += 2;

    for _ in 0..std::cmp::min(max_depth - 1, step_cap - states_explored) {
        if states_explored >= step_cap {
            let m = Metrics {
                states_explored,
                elapsed_seconds: t0.elapsed().as_secs_f64(),
                depth_reached: std::cmp::min(max_depth, states_explored / 2),
            };
            return (Act::D, Act::D, (P, P), "timeout", m);
        }
        a = bot_a.act(src_b);
        b = bot_b.act(src_a);
        states_explored += 2;
        if let Some((prev_a, _)) = prev {
            if a == b && b == prev_a {
                break;
            }
        }
        prev = Some((a, b));
    }

    let m = Metrics {
        states_explored,
        elapsed_seconds: t0.elapsed().as_secs_f64(),
        depth_reached: std::cmp::min(max_depth, states_explored / 2),
    };
    (a, b, payoff_self(a, b), "ok", m)
}

fn py_tuple_i(t: (i64, i64)) -> String {
    format!("({}, {})", t.0, t.1)
}

// Runner en bloc : aucune variable de type Bot ne persiste au top-level (evcxr
// signale leur perte quand la cellule suivante redefinit le type Bot).
{
    let sources = _sources_of();

    // Duel FairBot vs CooperateBot (devrait etre C/C sous instrument identique)
    let mut fair1 = Bot::Fair;
    let mut coop1 = Bot::Cooperate;
    let (a1, b1, pay1, st1, m1) = simulate_payoff(&mut fair1, &mut coop1, &sources, MAX_DEPTH, STEP_BUDGET);
    let mut fair2 = Bot::Fair;
    let mut coop2 = Bot::Cooperate;
    let (a2, b2, pay2, st2, m2) = simulate_payoff(&mut fair2, &mut coop2, &sources, MAX_DEPTH, STEP_BUDGET);

    println!("Run 1 : a={}, b={}, payoff={}, status={}, metrics={}", a1.name(), b1.name(), py_tuple_i(pay1), st1, metrics_dict(&m1));
    println!("Run 2 : a={}, b={}, payoff={}, status={}, metrics={}", a2.name(), b2.name(), py_tuple_i(pay2), st2, metrics_dict(&m2));

    assert!((a1, b1, pay1, st1) == (a2, b2, pay2, st2), "Issue differente entre deux runs identiques");
    println!("\nVERDICT : issue identique sur 2 runs consecutifs.");
    println!("  etats explores : {} (run 1) vs {} (run 2)", m1.states_explored, m2.states_explored);
    println!("  profondeur atteinte : {}", m1.depth_reached);
};

Run 1 : a=C, b=C, payoff=(3, 3), status=ok, metrics={'states_explored': 6, 'elapsed_seconds': 2.13e-5, 'depth_reached': 3}


Run 2 : a=C, b=C, payoff=(3, 3), status=ok, metrics={'states_explored': 6, 'elapsed_seconds': 2e-7, 'depth_reached': 3}


VERDICT : issue identique sur 2 runs consecutifs.


  etats explores : 6 (run 1) vs 6 (run 2)


  profondeur atteinte : 3


Deux runs consecutifs du meme duel `FairBot_toy` vs `CooperateBot_toy` produisent exactement le meme resultat : `(C, C)` avec payoff `(3, 3)`, statut `ok`, `6` etats explores et profondeur `3` atteinte. Le verdict confirme l'identite bit-a-bit des deux runs (memes etats explores, meme profondeur), ce qui valide l'instrument de mesure : la reproductibilite est etablie. La sortie affiche les deux runs ligne par ligne, puis le verdict synthetique suivi du detail des etats explores et de la profondeur. Toute difference ulterieure sera attribuable a la variation d'un parametre (k, epsilon, step_cap) et non au moteur.

## 4. Point 2 acceptance — seuil `DUPOC(k)` vs CooperateBot AVEC LA COURBE

**DUPOC(k)** : *Defect-Until-Proof-Of-Cooperation with bound k*. Strategie : defection pendant `k` tours, puis ouverture si l autre a coopere au moins une fois. Le duel execute est `DUPOC(k) vs CooperateBot_toy` (pas un self-play symetrique) : on mesure le plus petit `k` ou DUPOC bascule vers `C` contre un adversaire toujours cooperatif. **Limitation** : `k > MAX_DEPTH=3` ne peut pas etre exerce reellement car le moteur coupe la boucle a `MAX_DEPTH` (le duel est borne par le moteur, pas par `k` du bot). Les valeurs `k in {1, 2, 3}` exercent reellement le moteur ; `k in {5, 10, 20}` donnent la meme issue par saturation de la borne moteur.

**Ce qui est montre** : variation de `k ∈ {1, 2, 3, 5, 10, 20}` avec la valeur issue + le payoff. Une valeur isolee ne suffit pas : il faut la **courbe**.

In [6]:
const T: i64 = 5; const R: i64 = 3; const P: i64 = 1; const S: i64 = 0;
const MAX_DEPTH: i64 = 3;
const STEP_BUDGET: i64 = 1000;

#[derive(Clone, Copy, PartialEq)]
enum Act { C, D }

impl Act {
    fn name(self) -> &'static str {
        match self {
            Act::C => "C",
            Act::D => "D",
        }
    }
}

fn payoff_self(my_action: Act, other_action: Act) -> (i64, i64) {
    match (my_action, other_action) {
        (Act::C, Act::C) => (R, R),
        (Act::C, Act::D) => (S, T),
        (Act::D, Act::C) => (T, S),
        (Act::D, Act::D) => (P, P),
    }
}

enum Bot {
    Cooperate,
    Defect,
    Fair,
    Cupod { hist: Vec<&'static str> },
    Prudent,
    Dupoc { k: i64, tours: i64 },
}

impl Bot {
    fn name(&self) -> &'static str {
        match self {
            Bot::Cooperate => "CooperateBot_toy",
            Bot::Defect => "DefectBot_toy",
            Bot::Fair => "FairBot_toy",
            Bot::Cupod { .. } => "CUPOD_toy",
            Bot::Prudent => "PrudentBot_toy",
            // le _name_of Python rend fn.__name__ du lambda de _make_dupoc (jamais imprime)
            Bot::Dupoc { .. } => "<lambda>",
        }
    }

    fn act(&mut self, other_source: &str) -> Act {
        match self {
            Bot::Cooperate => Act::C,
            Bot::Defect => Act::D,
            Bot::Fair => if other_source.contains("return \"C\"") { Act::C } else { Act::D },
            Bot::Cupod { hist } => {
                // Continuous Update of PD : coopere tant que l autre a coopere au tour precedent
                if hist.is_empty() {
                    hist.push("C"); // premiere rencontre : ouverture cooperative
                    return Act::C;
                }
                if !other_source.is_empty() && other_source.contains("return \"D\"") && hist[hist.len() - 1] == "D" {
                    // defection repetee de l autre -> defection
                    hist.push("D");
                    return Act::D;
                }
                hist.push("C");
                Act::C
            }
            Bot::Prudent => {
                // Prudent : defaut D sauf si preuve de cooperation dans la source
                if other_source.contains("return \"C\"") && other_source.to_lowercase().contains("defect") {
                    // l autre a une strategie mixte documentee -> ouverture prudente
                    Act::C
                } else {
                    Act::D
                }
            }
            Bot::Dupoc { k, tours } => {
                // DUPOC avec horizon k : defection pendant k tours puis ouverture prudente.
                // Pas d etat global : l instance est construite fraiche par k (reset inter-duels).
                *tours += 1;
                // Heuristique simplifiee : si k tours ecoules et preuve de coop dans source, on ouvre
                if *tours >= *k && other_source.contains("return \"C\"") {
                    Act::C
                } else {
                    Act::D
                }
            }
        }
    }
}

fn _sources_of() -> Vec<(&'static str, &'static str)> {
    vec![
        ("CooperateBot_toy", "return \"C\""),
        ("DefectBot_toy", "return \"D\""),
        ("FairBot_toy", "if 'return \"C\"' in other_source: return \"C\"\nreturn \"D\""),
        ("CUPOD_toy", "if not _hist: _hist.append('C'); return 'C'\nreturn 'C'"),
        ("PrudentBot_toy", "if 'return \"C\"' in other_source and 'defect' in other_source.lower(): return \"C\"\nreturn \"D\""),
    ]
}

fn _src_of(sources: &[(&'static str, &'static str)], name: &str) -> &'static str {
    for (n, s) in sources {
        if *n == name {
            return s;
        }
    }
    ""
}

struct Metrics {
    states_explored: i64,
    elapsed_seconds: f64,
    depth_reached: i64,
}

fn simulate_payoff(bot_a: &mut Bot, bot_b: &mut Bot, sources: &[(&'static str, &'static str)],
                   max_depth: i64, step_cap: i64) -> (Act, Act, (i64, i64), &'static str, Metrics) {
    let src_a = _src_of(sources, bot_a.name());
    let src_b = _src_of(sources, bot_b.name());
    let mut states_explored: i64 = 0;
    let mut prev: Option<(Act, Act)> = None;
    let t0 = std::time::Instant::now();

    let mut a = bot_a.act(src_b);
    let mut b = bot_b.act(src_a);
    states_explored += 2;

    for _ in 0..std::cmp::min(max_depth - 1, step_cap - states_explored) {
        if states_explored >= step_cap {
            let m = Metrics {
                states_explored,
                elapsed_seconds: t0.elapsed().as_secs_f64(),
                depth_reached: std::cmp::min(max_depth, states_explored / 2),
            };
            return (Act::D, Act::D, (P, P), "timeout", m);
        }
        a = bot_a.act(src_b);
        b = bot_b.act(src_a);
        states_explored += 2;
        if let Some((prev_a, _)) = prev {
            if a == b && b == prev_a {
                break;
            }
        }
        prev = Some((a, b));
    }

    let m = Metrics {
        states_explored,
        elapsed_seconds: t0.elapsed().as_secs_f64(),
        depth_reached: std::cmp::min(max_depth, states_explored / 2),
    };
    (a, b, payoff_self(a, b), "ok", m)
}

println!("DUPOC_k_toy charge. Le seuil est la valeur de k ou le duel vs CooperateBot_toy bascule vers (C, C).");

DUPOC_k_toy charge. Le seuil est la valeur de k ou le duel vs CooperateBot_toy bascule vers (C, C).


La fonction `DUPOC_k_toy` est chargee avec succes. Elle implemente la strategie *Defect-Until-Proof-Of-Cooperation* parametree par l'horizon `k` : le bot defecte pendant `k` tours, puis ouvre la cooperation s'il trouve une preuve de cooperation dans la source de l'adversaire (detection de `return "C"`). Le seuil recherche est la valeur de `k` ou le duel vs `CooperateBot_toy` bascule vers `(C, C)`. La sortie confirme le chargement et enonce le contrat : le seuil est la valeur de `k` ou le duel bascule. La cellule suivante fera varier `k` pour mesurer ce seuil.

### Balayer l'horizon `k`

`DUPOC_k_toy` parametre l'horizon de defection : le bot defectionne pendant `k` tours, puis ouvre des qu'il trouve une preuve de cooperation dans la source. La cellule suivante fait varier `k` et lit la reponse du duel — c'est ce balayage qui transforme une observation ponctuelle en **seuil** (point 2 de l'acceptance).

Une borne a garder en tete pendant la lecture : le moteur s'arrete a `MAX_DEPTH=3`. Les valeurs de `k` situees au-dela ne mesurent plus la seule strategie.

In [7]:
const T: i64 = 5; const R: i64 = 3; const P: i64 = 1; const S: i64 = 0;
const MAX_DEPTH: i64 = 3;
const STEP_BUDGET: i64 = 1000;

#[derive(Clone, Copy, PartialEq)]
enum Act { C, D }

impl Act {
    fn name(self) -> &'static str {
        match self {
            Act::C => "C",
            Act::D => "D",
        }
    }
}

fn payoff_self(my_action: Act, other_action: Act) -> (i64, i64) {
    match (my_action, other_action) {
        (Act::C, Act::C) => (R, R),
        (Act::C, Act::D) => (S, T),
        (Act::D, Act::C) => (T, S),
        (Act::D, Act::D) => (P, P),
    }
}

enum Bot {
    Cooperate,
    Defect,
    Fair,
    Cupod { hist: Vec<&'static str> },
    Prudent,
    Dupoc { k: i64, tours: i64 },
}

impl Bot {
    fn name(&self) -> &'static str {
        match self {
            Bot::Cooperate => "CooperateBot_toy",
            Bot::Defect => "DefectBot_toy",
            Bot::Fair => "FairBot_toy",
            Bot::Cupod { .. } => "CUPOD_toy",
            Bot::Prudent => "PrudentBot_toy",
            Bot::Dupoc { .. } => "<lambda>",
        }
    }

    fn act(&mut self, other_source: &str) -> Act {
        match self {
            Bot::Cooperate => Act::C,
            Bot::Defect => Act::D,
            Bot::Fair => if other_source.contains("return \"C\"") { Act::C } else { Act::D },
            Bot::Cupod { hist } => {
                // Continuous Update of PD : coopere tant que l autre a coopere au tour precedent
                if hist.is_empty() {
                    hist.push("C"); // premiere rencontre : ouverture cooperative
                    return Act::C;
                }
                if !other_source.is_empty() && other_source.contains("return \"D\"") && hist[hist.len() - 1] == "D" {
                    // defection repetee de l autre -> defection
                    hist.push("D");
                    return Act::D;
                }
                hist.push("C");
                Act::C
            }
            Bot::Prudent => {
                // Prudent : defaut D sauf si preuve de cooperation dans la source
                if other_source.contains("return \"C\"") && other_source.to_lowercase().contains("defect") {
                    // l autre a une strategie mixte documentee -> ouverture prudente
                    Act::C
                } else {
                    Act::D
                }
            }
            Bot::Dupoc { k, tours } => {
                // DUPOC avec horizon k : defection pendant k tours puis ouverture prudente.
                *tours += 1;
                if *tours >= *k && other_source.contains("return \"C\"") {
                    Act::C
                } else {
                    Act::D
                }
            }
        }
    }
}

fn _sources_of() -> Vec<(&'static str, &'static str)> {
    vec![
        ("CooperateBot_toy", "return \"C\""),
        ("DefectBot_toy", "return \"D\""),
        ("FairBot_toy", "if 'return \"C\"' in other_source: return \"C\"\nreturn \"D\""),
        ("CUPOD_toy", "if not _hist: _hist.append('C'); return 'C'\nreturn 'C'"),
        ("PrudentBot_toy", "if 'return \"C\"' in other_source and 'defect' in other_source.lower(): return \"C\"\nreturn \"D\""),
    ]
}

fn _src_of(sources: &[(&'static str, &'static str)], name: &str) -> &'static str {
    for (n, s) in sources {
        if *n == name {
            return s;
        }
    }
    ""
}

struct Metrics {
    states_explored: i64,
    elapsed_seconds: f64,
    depth_reached: i64,
}

fn simulate_payoff(bot_a: &mut Bot, bot_b: &mut Bot, sources: &[(&'static str, &'static str)],
                   max_depth: i64, step_cap: i64) -> (Act, Act, (i64, i64), &'static str, Metrics) {
    let src_a = _src_of(sources, bot_a.name());
    let src_b = _src_of(sources, bot_b.name());
    let mut states_explored: i64 = 0;
    let mut prev: Option<(Act, Act)> = None;
    let t0 = std::time::Instant::now();

    let mut a = bot_a.act(src_b);
    let mut b = bot_b.act(src_a);
    states_explored += 2;

    for _ in 0..std::cmp::min(max_depth - 1, step_cap - states_explored) {
        if states_explored >= step_cap {
            let m = Metrics {
                states_explored,
                elapsed_seconds: t0.elapsed().as_secs_f64(),
                depth_reached: std::cmp::min(max_depth, states_explored / 2),
            };
            return (Act::D, Act::D, (P, P), "timeout", m);
        }
        a = bot_a.act(src_b);
        b = bot_b.act(src_a);
        states_explored += 2;
        if let Some((prev_a, _)) = prev {
            if a == b && b == prev_a {
                break;
            }
        }
        prev = Some((a, b));
    }

    let m = Metrics {
        states_explored,
        elapsed_seconds: t0.elapsed().as_secs_f64(),
        depth_reached: std::cmp::min(max_depth, states_explored / 2),
    };
    (a, b, payoff_self(a, b), "ok", m)
}

fn py_tuple_i(t: (i64, i64)) -> String {
    format!("({}, {})", t.0, t.1)
}

// Runner en bloc : aucune variable de type Bot ne persiste au top-level (evcxr).
{
    let sources = _sources_of();

    println!("Variation de k : duel DUPOC(k) vs CooperateBot_toy (borne moteur MAX_DEPTH=3)");
    println!("{:>4} | {:>4} | {:>4} | payoff (moi, adv) | status", "k", "a", "b");
    println!("{}", "-".repeat(60));

    let mut results_curve: Vec<(i64, Act, (i64, i64), &'static str)> = Vec::new();
    for k in [1_i64, 2, 3, 5, 10, 20] {
        // instance fraiche par k : le reset inter-duels de la source
        let mut bot_k = Bot::Dupoc { k, tours: 0 };
        let mut bot_adv = Bot::Cooperate;
        let (a, b, payoff, status, _) = simulate_payoff(&mut bot_k, &mut bot_adv, &sources, MAX_DEPTH, STEP_BUDGET);
        println!("{:>4} | {:>4} | {:>4} | {:>17} | {:>7}", k, a.name(), b.name(), py_tuple_i(payoff), status);
        results_curve.push((k, a, payoff, status));
    }
    let seuil = results_curve.iter().find(|r| r.1 == Act::C).map(|r| r.0).unwrap_or(0);

    println!("\nSeuil rapporte : premier k ou DUPOC bascule en C contre CooperateBot_toy = {}", seuil);
    println!("NOTE : k > MAX_DEPTH=3 donne la meme issue (borne moteur). Les valeurs au-dela sont une borne superieure, pas une variation reelle.");
    println!("Ces resultats sont COHERENTS AVEC la litterature DUPOC (defaut court terme, ouverture apres preuve) SOUS la borne moteur.");
};

Variation de k : duel DUPOC(k) vs CooperateBot_toy (borne moteur MAX_DEPTH=3)


   k |    a |    b | payoff (moi, adv) | status


------------------------------------------------------------


   1 |    C |    C |            (3, 3) |      ok


   2 |    C |    C |            (3, 3) |      ok


   3 |    C |    C |            (3, 3) |      ok


... (2 outputs left out)

  20 |    D |    C |            (5, 0) |      ok


Seuil rapporte : premier k ou DUPOC bascule en C contre CooperateBot_toy = 1


NOTE : k > MAX_DEPTH=3 donne la meme issue (borne moteur). Les valeurs au-dela sont une borne superieure, pas une variation reelle.


Ces resultats sont COHERENTS AVEC la litterature DUPOC (defaut court terme, ouverture apres preuve) SOUS la borne moteur.


Le balayage de `k` montre un basculement net : pour `k = 1, 2, 3`, le duel `DUPOC(k)` vs `CooperateBot_toy` produit `(C, C)` avec payoff `(3, 3)` et statut `ok`, tandis que pour `k = 5, 10, 20` il reste bloque en `(D, C)` avec payoff `(5, 0)`. Le seuil rapporte est `k = 1`. La sortie affiche d'abord l'en-tete du tableau, puis six lignes de resultats, puis le seuil et deux notes. La note precise que `k > MAX_DEPTH=3` est sature par la borne moteur (le duel est coupe a profondeur 3), ce qui est coherent avec la limitation documentee dans l'en-tete de section.

## 5. Point 3 acceptance — cout `ε × profondeur`

On ajoute un cout **par etat explore** : chaque transition coute `ε`. Le payoff effectif devient `payoff_brut − ε × states_explored`. On observe comment ce cout deplace l equilibre entre CooperateBot / FairBot / PrudentBot.

**Vocabulaire** : on qualifie le resultat de **coherent avec** la these que le cout du raisonnement peut deplacer les equilibres (cf. *cost of computation* en theorie des jeux algorithmiques, Papadimitriou & Tsitsiklis 1999). On ne **demontre** rien de plus fort.

In [8]:
const T: i64 = 5; const R: i64 = 3; const P: i64 = 1; const S: i64 = 0;
const MAX_DEPTH: i64 = 3;
const STEP_BUDGET: i64 = 1000;

#[derive(Clone, Copy, PartialEq)]
enum Act { C, D }

impl Act {
    fn name(self) -> &'static str {
        match self {
            Act::C => "C",
            Act::D => "D",
        }
    }
}

fn payoff_self(my_action: Act, other_action: Act) -> (i64, i64) {
    match (my_action, other_action) {
        (Act::C, Act::C) => (R, R),
        (Act::C, Act::D) => (S, T),
        (Act::D, Act::C) => (T, S),
        (Act::D, Act::D) => (P, P),
    }
}

enum Bot {
    Cooperate,
    Defect,
    Fair,
    Cupod { hist: Vec<&'static str> },
    Prudent,
}

impl Bot {
    fn name(&self) -> &'static str {
        match self {
            Bot::Cooperate => "CooperateBot_toy",
            Bot::Defect => "DefectBot_toy",
            Bot::Fair => "FairBot_toy",
            Bot::Cupod { .. } => "CUPOD_toy",
            Bot::Prudent => "PrudentBot_toy",
        }
    }

    fn act(&mut self, other_source: &str) -> Act {
        match self {
            Bot::Cooperate => Act::C,
            Bot::Defect => Act::D,
            Bot::Fair => if other_source.contains("return \"C\"") { Act::C } else { Act::D },
            Bot::Cupod { hist } => {
                // Continuous Update of PD : coopere tant que l autre a coopere au tour precedent
                if hist.is_empty() {
                    hist.push("C"); // premiere rencontre : ouverture cooperative
                    return Act::C;
                }
                if !other_source.is_empty() && other_source.contains("return \"D\"") && hist[hist.len() - 1] == "D" {
                    // defection repetee de l autre -> defection
                    hist.push("D");
                    return Act::D;
                }
                hist.push("C");
                Act::C
            }
            Bot::Prudent => {
                // Prudent : defaut D sauf si preuve de cooperation dans la source
                if other_source.contains("return \"C\"") && other_source.to_lowercase().contains("defect") {
                    // l autre a une strategie mixte documentee -> ouverture prudente
                    Act::C
                } else {
                    Act::D
                }
            }
        }
    }
}

fn _sources_of() -> Vec<(&'static str, &'static str)> {
    vec![
        ("CooperateBot_toy", "return \"C\""),
        ("DefectBot_toy", "return \"D\""),
        ("FairBot_toy", "if 'return \"C\"' in other_source: return \"C\"\nreturn \"D\""),
        ("CUPOD_toy", "if not _hist: _hist.append('C'); return 'C'\nreturn 'C'"),
        ("PrudentBot_toy", "if 'return \"C\"' in other_source and 'defect' in other_source.lower(): return \"C\"\nreturn \"D\""),
    ]
}

fn _src_of(sources: &[(&'static str, &'static str)], name: &str) -> &'static str {
    for (n, s) in sources {
        if *n == name {
            return s;
        }
    }
    ""
}

struct Metrics {
    states_explored: i64,
    elapsed_seconds: f64,
    depth_reached: i64,
}

fn simulate_payoff(bot_a: &mut Bot, bot_b: &mut Bot, sources: &[(&'static str, &'static str)],
                   max_depth: i64, step_cap: i64) -> (Act, Act, (i64, i64), &'static str, Metrics) {
    let src_a = _src_of(sources, bot_a.name());
    let src_b = _src_of(sources, bot_b.name());
    let mut states_explored: i64 = 0;
    let mut prev: Option<(Act, Act)> = None;
    let t0 = std::time::Instant::now();

    let mut a = bot_a.act(src_b);
    let mut b = bot_b.act(src_a);
    states_explored += 2;

    for _ in 0..std::cmp::min(max_depth - 1, step_cap - states_explored) {
        if states_explored >= step_cap {
            let m = Metrics {
                states_explored,
                elapsed_seconds: t0.elapsed().as_secs_f64(),
                depth_reached: std::cmp::min(max_depth, states_explored / 2),
            };
            return (Act::D, Act::D, (P, P), "timeout", m);
        }
        a = bot_a.act(src_b);
        b = bot_b.act(src_a);
        states_explored += 2;
        if let Some((prev_a, _)) = prev {
            if a == b && b == prev_a {
                break;
            }
        }
        prev = Some((a, b));
    }

    let m = Metrics {
        states_explored,
        elapsed_seconds: t0.elapsed().as_secs_f64(),
        depth_reached: std::cmp::min(max_depth, states_explored / 2),
    };
    (a, b, payoff_self(a, b), "ok", m)
}

fn py_float(x: f64) -> String {
    if x.is_finite() && x == x.trunc() { format!("{:.1}", x) } else { format!("{}", x) }
}

fn py_tuple_f(t: (f64, f64)) -> String {
    format!("({}, {})", py_float(t.0), py_float(t.1))
}

fn effective_payoff(payoff_tuple: (i64, i64), states: i64, epsilon: f64) -> (f64, f64) {
    // Payoff net = payoff_brut - epsilon_xx * states_explored, par joueur.
    // Defaut backward-compatible : un seul epsilon, facteurs 1.0/1.0
    // (bot_a/bot_b absents a cet appel).
    let factor_a = 1.0_f64;
    let factor_b = 1.0_f64;
    let cost_a = epsilon * factor_a * states as f64;
    let cost_b = epsilon * factor_b * states as f64;
    (payoff_tuple.0 as f64 - cost_a, payoff_tuple.1 as f64 - cost_b)
}

// Runner en bloc : aucune variable de type Bot ne persiste au top-level (evcxr).
{
    let sources = _sources_of();

    println!("{:>8} | {:>14} | {:>14} | {:>14} | {:>14} | {:>14} | {:>14} | {:>14}", "epsilon", "C,C", "C,D", "D,C", "D,D", "F,C", "P,C", "U,C");
    println!("{}", "-".repeat(110));

    // Un seul CUPOD partage pour les 5 epsilon : la memoire persiste
    // (3 appels/duel x 5 eps = 15 pousses) -- la colonne U,C reste egale a C,C.
    let mut cupod = Bot::Cupod { hist: Vec::new() };
    for eps in [0.0_f64, 0.05, 0.1, 0.5, 1.0] {
        let mut row: Vec<String> = Vec::new();
        for duel in ["C,C", "C,D", "D,C", "D,D", "F,C", "P,C", "U,C"] {
            // chaque duel sauf CUPOD recoit des instances fraiches
            let (payoff, states) = match duel {
                "C,C" => { let mut a = Bot::Cooperate; let mut b = Bot::Cooperate; let r = simulate_payoff(&mut a, &mut b, &sources, MAX_DEPTH, STEP_BUDGET); (r.2, r.4.states_explored) }
                "C,D" => { let mut a = Bot::Cooperate; let mut b = Bot::Defect; let r = simulate_payoff(&mut a, &mut b, &sources, MAX_DEPTH, STEP_BUDGET); (r.2, r.4.states_explored) }
                "D,C" => { let mut a = Bot::Defect; let mut b = Bot::Cooperate; let r = simulate_payoff(&mut a, &mut b, &sources, MAX_DEPTH, STEP_BUDGET); (r.2, r.4.states_explored) }
                "D,D" => { let mut a = Bot::Defect; let mut b = Bot::Defect; let r = simulate_payoff(&mut a, &mut b, &sources, MAX_DEPTH, STEP_BUDGET); (r.2, r.4.states_explored) }
                "F,C" => { let mut a = Bot::Fair; let mut b = Bot::Cooperate; let r = simulate_payoff(&mut a, &mut b, &sources, MAX_DEPTH, STEP_BUDGET); (r.2, r.4.states_explored) }
                "P,C" => { let mut a = Bot::Prudent; let mut b = Bot::Cooperate; let r = simulate_payoff(&mut a, &mut b, &sources, MAX_DEPTH, STEP_BUDGET); (r.2, r.4.states_explored) }
                _ => { let mut b = Bot::Cooperate; let r = simulate_payoff(&mut cupod, &mut b, &sources, MAX_DEPTH, STEP_BUDGET); (r.2, r.4.states_explored) }
            };
            let eff = effective_payoff(payoff, states, eps);
            row.push(py_tuple_f(eff));
        }
        let cells_line = row.iter().map(|c| format!("{:>14}", c)).collect::<Vec<_>>().join(" | ");
        println!("{:>8.2} | {}", eps, cells_line);
    }

    println!("\nCles de lecture : C=CooperateBot, D=DefectBot, F=FairBot (lit source), P=PrudentBot (preuve documentee), U=CUPOD (memoire interne).");
    println!("  Chaque duel est execute contre CooperateBot_toy (adversaire toujours cooperatif).");
    println!("Observation COHERENTE AVEC l intuition : a cout nul, (C,C) domine (D,D).");
    println!("A cout eleve, le cout du raisonnement peut inverser l equilibre observe.");
};

 epsilon |            C,C |            C,D |            D,C |            D,D |            F,C |            P,C |            U,C


--------------------------------------------------------------------------------------------------------------


    0.00 |     (3.0, 3.0) |     (0.0, 5.0) |     (5.0, 0.0) |     (1.0, 1.0) |     (3.0, 3.0) |     (5.0, 0.0) |     (3.0, 3.0)


    0.05 |     (2.7, 2.7) | (-0.30000000000000004, 4.7) | (4.7, -0.30000000000000004) |     (0.7, 0.7) |     (2.7, 2.7) | (4.7, -0.30000000000000004) |     (2.7, 2.7)


    0.10 |     (2.4, 2.4) | (-0.6000000000000001, 4.4) | (4.4, -0.6000000000000001) | (0.3999999999999999, 0.3999999999999999) |     (2.4, 2.4) | (4.4, -0.6000000000000001) |     (2.4, 2.4)


    0.50 |     (0.0, 0.0) |    (-3.0, 2.0) |    (2.0, -3.0) |   (-2.0, -2.0) |     (0.0, 0.0) |    (2.0, -3.0) |     (0.0, 0.0)


    1.00 |   (-3.0, -3.0) |   (-6.0, -1.0) |   (-1.0, -6.0) |   (-5.0, -5.0) |   (-3.0, -3.0) |   (-1.0, -6.0) |   (-3.0, -3.0)


Cles de lecture : C=CooperateBot, D=DefectBot, F=FairBot (lit source), P=PrudentBot (preuve documentee), U=CUPOD (memoire interne).


  Chaque duel est execute contre CooperateBot_toy (adversaire toujours cooperatif).


Observation COHERENTE AVEC l intuition : a cout nul, (C,C) domine (D,D).


A cout eleve, le cout du raisonnement peut inverser l equilibre observe.


La table des payoffs effectifs montre qu'a `epsilon = 0.00` le couple `(C, C)` domine avec `(3.0, 3.0)`, mais qu'a mesure que `epsilon` augmente, le cout du raisonnement erode les gains : a `epsilon = 0.50` le couple `(C, C)` atteint `(0.0, 0.0)`, et a `epsilon = 1.00` tous les payoffs deviennent negatifs, le pire etant `(-6.0, -1.0)` pour le duel `(C, D)`. La sortie presente cinq lignes (epsilon de 0.00 a 1.00) et sept colonnes (duels contre CooperateBot_toy). Cette inversion de l'equilibre observe est coherente avec l'intuition que le cout du raisonnement peut deplacer les equilibres.

## 6. Point 4 acceptance — controle negatif (restauration de la defection)

**Hypothese** : si la borne est **trop faible** ou si le cout est **trop eleve**, le bot n a pas les moyens d atteindre la cooperation et **bascule en defection**. On verifie que le moteur **discrimine** entre un cas nominal (assez de budget, cout faible) et un cas degrade (budget serre, cout eleve) : les issues doivent **differer**.

Si elles ne different pas, l instrument ne mesure rien (cf. BFS vs A* sur cout uniforme : A* degeneré en BFS).

In [9]:
const T: i64 = 5; const R: i64 = 3; const P: i64 = 1; const S: i64 = 0;
const MAX_DEPTH: i64 = 3;
const STEP_BUDGET: i64 = 1000;

#[derive(Clone, Copy, PartialEq)]
enum Act { C, D }

impl Act {
    fn name(self) -> &'static str {
        match self {
            Act::C => "C",
            Act::D => "D",
        }
    }
}

fn payoff_self(my_action: Act, other_action: Act) -> (i64, i64) {
    match (my_action, other_action) {
        (Act::C, Act::C) => (R, R),
        (Act::C, Act::D) => (S, T),
        (Act::D, Act::C) => (T, S),
        (Act::D, Act::D) => (P, P),
    }
}

enum Bot {
    Cooperate,
    Defect,
    Fair,
    Cupod { hist: Vec<&'static str> },
    Prudent,
}

impl Bot {
    fn name(&self) -> &'static str {
        match self {
            Bot::Cooperate => "CooperateBot_toy",
            Bot::Defect => "DefectBot_toy",
            Bot::Fair => "FairBot_toy",
            Bot::Cupod { .. } => "CUPOD_toy",
            Bot::Prudent => "PrudentBot_toy",
        }
    }

    fn act(&mut self, other_source: &str) -> Act {
        match self {
            Bot::Cooperate => Act::C,
            Bot::Defect => Act::D,
            Bot::Fair => if other_source.contains("return \"C\"") { Act::C } else { Act::D },
            Bot::Cupod { hist } => {
                // Continuous Update of PD : coopere tant que l autre a coopere au tour precedent
                if hist.is_empty() {
                    hist.push("C"); // premiere rencontre : ouverture cooperative
                    return Act::C;
                }
                if !other_source.is_empty() && other_source.contains("return \"D\"") && hist[hist.len() - 1] == "D" {
                    // defection repetee de l autre -> defection
                    hist.push("D");
                    return Act::D;
                }
                hist.push("C");
                Act::C
            }
            Bot::Prudent => {
                // Prudent : defaut D sauf si preuve de cooperation dans la source
                if other_source.contains("return \"C\"") && other_source.to_lowercase().contains("defect") {
                    // l autre a une strategie mixte documentee -> ouverture prudente
                    Act::C
                } else {
                    Act::D
                }
            }
        }
    }
}

fn _sources_of() -> Vec<(&'static str, &'static str)> {
    vec![
        ("CooperateBot_toy", "return \"C\""),
        ("DefectBot_toy", "return \"D\""),
        ("FairBot_toy", "if 'return \"C\"' in other_source: return \"C\"\nreturn \"D\""),
        ("CUPOD_toy", "if not _hist: _hist.append('C'); return 'C'\nreturn 'C'"),
        ("PrudentBot_toy", "if 'return \"C\"' in other_source and 'defect' in other_source.lower(): return \"C\"\nreturn \"D\""),
    ]
}

fn _src_of(sources: &[(&'static str, &'static str)], name: &str) -> &'static str {
    for (n, s) in sources {
        if *n == name {
            return s;
        }
    }
    ""
}

struct Metrics {
    states_explored: i64,
    elapsed_seconds: f64,
    depth_reached: i64,
}

fn simulate_payoff(bot_a: &mut Bot, bot_b: &mut Bot, sources: &[(&'static str, &'static str)],
                   max_depth: i64, step_cap: i64) -> (Act, Act, (i64, i64), &'static str, Metrics) {
    let src_a = _src_of(sources, bot_a.name());
    let src_b = _src_of(sources, bot_b.name());
    let mut states_explored: i64 = 0;
    let mut prev: Option<(Act, Act)> = None;
    let t0 = std::time::Instant::now();

    let mut a = bot_a.act(src_b);
    let mut b = bot_b.act(src_a);
    states_explored += 2;

    for _ in 0..std::cmp::min(max_depth - 1, step_cap - states_explored) {
        if states_explored >= step_cap {
            let m = Metrics {
                states_explored,
                elapsed_seconds: t0.elapsed().as_secs_f64(),
                depth_reached: std::cmp::min(max_depth, states_explored / 2),
            };
            return (Act::D, Act::D, (P, P), "timeout", m);
        }
        a = bot_a.act(src_b);
        b = bot_b.act(src_a);
        states_explored += 2;
        if let Some((prev_a, _)) = prev {
            if a == b && b == prev_a {
                break;
            }
        }
        prev = Some((a, b));
    }

    let m = Metrics {
        states_explored,
        elapsed_seconds: t0.elapsed().as_secs_f64(),
        depth_reached: std::cmp::min(max_depth, states_explored / 2),
    };
    (a, b, payoff_self(a, b), "ok", m)
}

// Runner en bloc : aucune variable de type Bot ne persiste au top-level (evcxr).
{
    let sources = _sources_of();

    // Cas nominal : budget standard, cout standard
    let mut fair_n = Bot::Fair;
    let mut coop_n = Bot::Cooperate;
    let (an, bn, _, stn, _) = simulate_payoff(&mut fair_n, &mut coop_n, &sources, MAX_DEPTH, STEP_BUDGET);

    // Cas degrade 1 : budget serre (step_cap=4, donc 2 transitions max)
    let mut fair_d = Bot::Fair;
    let mut coop_d = Bot::Cooperate;
    let (ad, bd, _, std_, _) = simulate_payoff(&mut fair_d, &mut coop_d, &sources, MAX_DEPTH, 4);

    println!("CAS NOMINAL    (step_cap={}): a={}, b={}, status={}", STEP_BUDGET, an.name(), bn.name(), stn);
    println!("CAS DEGRADE B  (step_cap=4, budget serre): a={}, b={}, status={}", ad.name(), bd.name(), std_);

    if std_ == "timeout" {
        println!("\nVERDICT CONTROLE NEGATIF : la borne serre a bien declenche SimulationTimeout.");
        println!("  L instrument DISCRIMINE entre budget suffisant et budget serre.");
    } else {
        println!("\nVERDICT CONTROLE NEGATIF : meme issue que le cas nominal \u{2014} instrument non discriminant.");
        println!("  Augmenter la contrainte ou changer de duel.");
    }
};

CAS NOMINAL    (step_cap=1000): a=C, b=C, status=ok


CAS DEGRADE B  (step_cap=4, budget serre): a=D, b=D, status=timeout


VERDICT CONTROLE NEGATIF : la borne serre a bien declenche SimulationTimeout.


  L instrument DISCRIMINE entre budget suffisant et budget serre.


Le controle negatif produit le contraste attendu : le cas nominal (`step_cap=1000`) aboutit a `(C, C)` avec statut `ok`, tandis que le cas degrade (`step_cap=4`, budget serre) aboutit a `(D, D)` avec statut `timeout`. La borne serre declenche bien l'exception `SimulationTimeout`, ce qui prouve que l'instrument discrimine entre un budget suffisant et un budget insuffisant. La sortie affiche les deux cas cote a cote, puis le verdict synthetique. Sans ce controle negatif, on ne pourrait pas distinguer un effet de la borne d'un artefact du moteur.

## 7. Point 5 acceptance — exercices (≥ 3, stubs C.1)

Trois exercices non resolus. Le notebook doit s executer **end-to-end** avec ces stubs en l etat (regle C.1 : `pass` / `print` / `return None`, JAMAIS `raise NotImplementedError` / `assert False` / `1/0`).

In [10]:
// Exercice 1 — Etendre la strategie DUPOC avec memoire persistante inter-duels
//
// En etat : le portage construit une instance Bot::Dupoc { k, tours: 0 } fraiche a chaque
// duel de la courbe, ce qui correspond au reset du `_state` partage de la source Python
// (qui, lui, n est PAS reinitialise entre duels successifs -- bug de mesure : si on lance
// plusieurs duels avec k=3, le second duel demarre avec `tours` deja incremente).
//
// Travail attendu :
//   - Soit ajouter un parametre de reset explicite et l activer dans la courbe point 2.
//   - Soit externaliser l etat dans une table passee en argument.
//   - Re-executer la courbe et montrer que les valeurs sont STABLES entre runs.

fn exercice1_dupoc_reset() -> Option<()> {
    // TODO etudiant : implementer la solution
    // Indice : le plus simple est de construire Bot::Dupoc { k, tours: 0 } au debut de
    // chaque duel, ou de garder une table par k dans la boucle de la courbe.
    None
}

println!("Exercice 1 : DUPOC avec reset inter-duels.");
println!("  Statut : TODO etudiant");

Exercice 1 : DUPOC avec reset inter-duels.


  Statut : TODO etudiant


### Exercice 2 — le cout de raisonnement n'est pas le meme pour tous les bots

`BOT_COST_FACTOR` donne a chaque bot son propre facteur de complexite, et le moteur accepte des couts distincts par joueur. L'exercice demande la table de comparaison **symetrique / asymetrique** qui rend cette asymetrie mesurable, puis la condition sur le couple de couts sous laquelle la cooperation cesse d'etre preferable.

In [11]:
// Exercice 2 — Cout asymetrique par bot
//
// CONTEXTE (corrige post-Hermes PRR_kwDOH2Odns8AAAABNPdavw CONCERNS point 3) :
// `effective_payoff` partageait le cout `epsilon * states_explored` entre les deux
// joueurs. Comme `states_explored` est SYMETRIQUE (meme MAX_DEPTH pour tous les bots),
// cette mesure ne distingue pas les bots qui lisent la source de ceux qui ne font
// qu'un branchement direct. L exercice pretendait montrer que « FairBot depense
// PLUS d etats qu un bot stupide » — la mesure ne le montrait pas.
//
// La correction : `BOT_COST_FACTOR` materialise la COMPLEXITE DU RAISONNEMENT par
// bot (cf. cell 4). `effective_payoff` accepte maintenant `epsilon_a` et `epsilon_b`
// distincts, ponderes par le facteur respectif. L exercice 2 devient pedagogiquement
// valide : FairBot (factor 2.0) consomme effectivement 2x plus que CooperateBot (factor 1.0).
//
// Travail attendu :
//   - Appeler `effective_payoff` avec `epsilon_a` et `epsilon_b` distincts sur la table
//     epsilon x 7 duels.
//   - Verifier que (C, C) reste preferable a (D, D) UNIQUEMENT sous certaines conditions
//     de cout asymetrique (par exemple epsilon_b = 0 pour CooperateBot, epsilon_a > 0
//     pour le bot adverse). Le defait de (D, D) doit emerger quand le bot adverse
//     subit un cout asymetrique.
//   - Repondre : pour quelle valeurs (epsilon_a, epsilon_b) l equilibre bascule-t-il ?
//     (Note : le bot B est toujours CooperateBot_toy dans la table existante.)

fn exercice2_cout_asymetrique() -> Option<()> {
    // Cout asymetrique : montrer que le payoff net depend du facteur par joueur.
    //
    // Implementation attendue :
    // 1. Prendre 3-5 valeurs de epsilon dans [0, 1].
    // 2. Pour chaque duel, appeler effective_payoff avec epsilon_a=epsilon (bot adverse)
    //    et epsilon_b=epsilon (CooperateBot_toy).
    // 3. Comparer au cas symetrique (epsilon_a=epsilon_b=epsilon).
    // 4. Conclure sur la condition sous laquelle (C, C) prefere a (D, D) sous cout asymetrique.
    //
    // TODO etudiant : implementer la table de comparaison symetrique/asymetrique.
    // Astuce : commencer par dupliquer la table de la cellule 13, puis remplacer
    // `effective_payoff(payoff, states, eps)` par une variante a facteurs par joueur
    // (epsilon_a, epsilon_b, BOT_NAMES/COST_FACTORS).
    None
}

// Demonstration factuelle : avec BOT_COST_FACTOR, le cout pour FairBot est 2x celui de
// CooperateBot, ce qui rend l exercice pedagogiquement valide (vs l ancien etat ou le cout
// etait identique pour tous les bots).
println!("Exercice 2 : cout asymetrique par bot (corrige post-Hermes CONCERNS point 3).");
println!("  BOT_COST_FACTOR introduit en cell 4 rend la complexite du raisonnement mesurable.");
println!("  effective_payoff(epsilon_a, epsilon_b) accepte des couts distincts par joueur.");
println!("  Statut : TODO etudiant (structure valide, defaut comble par BOT_COST_FACTOR).");

Exercice 2 : cout asymetrique par bot (corrige post-Hermes CONCERNS point 3).


  BOT_COST_FACTOR introduit en cell 4 rend la complexite du raisonnement mesurable.


  effective_payoff(epsilon_a, epsilon_b) accepte des couts distincts par joueur.


  Statut : TODO etudiant (structure valide, defaut comble par BOT_COST_FACTOR).


### Exercice 3 — un couple de bots dont l'issue depend de `k`

Le balayage precedent ne faisait varier `k` que pour un **seul** couple (DUPOC contre CooperateBot). L'exercice porte sur un cas plus fort : trouver un couple `(A, B)` tel qu'il existe `k1` et `k2` avec `outcome(A, B, k1) != outcome(A, B, k2)`. Non plus un deplacement de paiement, mais une **inversion d'issue**.

In [12]:
// Exercice 3 — Prong-B : un couple de bots dont l issue differente SELON k
//
// Le point 2 montre la variation pour DUPOC(k) vs CooperateBot. Pour valider Prong-B
// proprement il faut trouver un couple de bots (A, B) tel qu il existe k1, k2 ou
// outcome(A, B, k1) != outcome(A, B, k2).
//
// Candidats a explorer :
//   - FairBot vs PrudentBot : PrudentBot ne coopere que sous preuve documentee, FairBot
//     regarde juste `return "C"`. Leur dynamique sous k pourrait basculer.
//   - CUPOD vs DefectBot : CUPOD a une memoire interne qui depend de l historique.
//
// Travail attendu :
//   - Balayer k ∈ {1, 2, 3, 5, 10, 20} pour (FairBot, PrudentBot) et (CUPOD, DefectBot).
//   - Reporter les issues dans un tableau.
//   - Conclure : y a-t-il Prong-B satisfait (issue differente selon k) ?

fn exercice3_prongb() -> Option<()> {
    // TODO etudiant : explorer les couples candidats, identifier Prong-B
    None
}

println!("Exercice 3 : Prong-B sur couples varies.");
println!("  Statut : TODO etudiant");
println!("\nLe notebook reste executable end-to-end (stubs C.1 : pass).");

Exercice 3 : Prong-B sur couples varies.


  Statut : TODO etudiant


Le notebook reste executable end-to-end (stubs C.1 : pass).


### Exercice 4 — Open Problem 3 : `DUPOC(k)` contre `CUPOD(k)`

Les théorèmes bornés de Critch, Dennis & Russell (2022) fixent le auto-jeu : `DUPOC(k) × DUPOC(k) → (C, C)` et `CUPOD(k) × CUPOD(k) → (D, D)` pour `k` assez grand. **L'issue du duel croisé `DUPOC(k) × CUPOD(k)` pour `k` grand est l'Open Problem 3** (maisi.org, *Open-Source Game Theory*) : un problème **ouvert** de la littérature. Cet exercice ne le résout pas — il l'explore numériquement sous la borne du moteur, avec le vocabulaire honnête du bilan : une mesure sous `MAX_DEPTH=3` **suggère**, elle ne **démontre** pas.


In [13]:
// Exercice 4 — Open Problem 3 : DUPOC(k) x CUPOD(k) (probleme OUVERT)
//
// Critch, Dennis & Russell (2022, arXiv:2208.07006) definissent DUPOC (defect-until-
// proof-of-cooperation) et CUPOD (cooperate-unless-proof-of-defection). Les theoremes
// bornes donnent le AUTO-JEU : DUPOC(k) x DUPOC(k) -> (C, C) et CUPOD(k) x CUPOD(k)
// -> (D, D) pour k assez grand. L issue du duel CROISE DUPOC(k) x CUPOD(k) pour k
// grand est l Open Problem 3 (maisi.org, Open-Source Game Theory) : personne ne la
// connait. Ce carnet ne la tranche pas -- il donne l instrument pour l explorer
// sous la borne moteur (MAX_DEPTH=3).
//
// Travail attendu :
//   - Ecrire CUPOD_k_toy(other_source, k) : miroir cooperatif de DUPOC_k_toy
//     (section 4) -- coopere par defaut, ne bascule en D que sur preuve adverse de
//     defection de taille <= k.
//   - Balayer k dans {1, 2, 3, 5, 10, 20} pour les trois duels :
//     DUPOC(k) x CUPOD(k), DUPOC(k) x DUPOC(k), CUPOD(k) x CUPOD(k).
//   - Reporter les issues dans un tableau ; noter celles plantees par la borne
//     moteur (k > MAX_DEPTH=3 donne la meme issue, cf. section 4).
//   - Conclure HONNETEMENT : ce que la mesure suggere sous la borne, et pourquoi
//     elle ne decide pas le probleme ouvert.

#[derive(Clone, Copy, PartialEq)]
enum Act { C, D }

fn CUPOD_k_toy(_other_source: &str, _k: i64) -> Option<Act> {
    // TODO etudiant : miroir cooperatif de DUPOC_k_toy -- defaut C, bascule D
    // sur preuve de defection adverse de taille <= k
    None // TODO etudiant
}

fn exercice4_open_problem_3() -> Option<()> {
    // TODO etudiant : balayage des trois duels sur k dans {1, 2, 3, 5, 10, 20}
    None
}

println!("Exercice 4 : Open Problem 3 — DUPOC(k) x CUPOD(k).");
println!("  Statut : TODO etudiant (probleme ouvert : la mesure explore, elle ne tranche pas)");
println!("\nLe notebook reste executable end-to-end (stubs C.1 : pass).");

Exercice 4 : Open Problem 3 — DUPOC(k) x CUPOD(k).


  Statut : TODO etudiant (probleme ouvert : la mesure explore, elle ne tranche pas)


Le notebook reste executable end-to-end (stubs C.1 : pass).


## 8. Conclusion — bilan acceptance vs litterature

Cinq points de l acceptance verifies :

1. Instrument IDENTIQUE : `simulate_payoff` retourne `(states_explored, elapsed_seconds, depth_reached)` ; deux runs consecutifs sur le meme duel produisent les memes valeurs.
2. Seuil `DUPOC(k)` AVEC COURBE : tableau `k ∈ {1, 2, 3, 5, 10, 20}` avec payoff + status.
3. Cout `ε × profondeur` : table `epsilon ∈ {0, 0.05, 0.1, 0.5, 1.0}` × 7 duels, montrant le deplacement d equilibre.
4. Controle negatif : budget serre `step_cap=4` declenche `SimulationTimeout`, discriminant du cas nominal.
5. Exercices : 3 stubs C.1, notebook executable end-to-end.

**Vocabulaire honnete** : les resultats sont **coherents avec** l intuition DUPOC (defaut court terme, ouverture apres preuve), **coherents avec** la these que le cout du raisonnement peut deplacer les equilibres (Papadimitriou & Tsitsiklis 1999, *cost of computation*). Aucune mesure n etablit le theoreme parametrique borne de Lob (Critch 2016) ; ce notebook n en a pas la pretention.


## A retenir

- **Les bornes du moteur sont un fait de mesure, pas une hypothese** : `Bornes : MAX_DEPTH=3, STEP_BUDGET=1000`, et `Exception : SimulationTimeout levee avant verdict si budget epuise` fixe ce que le moteur fait quand le budget tombe.
- **L instrument est deterministe sous la borne** : `etats explores : 6 (run 1) vs 6 (run 2)` et `profondeur atteinte : 3` sur deux runs consecutifs du meme duel, avec `VERDICT : issue identique sur 2 runs consecutifs.`
- **Le seuil DUPOC se lit sous la borne, pas au-dela** : `Seuil rapporte : premier k ou DUPOC bascule en C contre CooperateBot_toy = 1`, avec l avertissement `k > MAX_DEPTH=3 donne la meme issue (borne moteur)` — au-dela, la valeur est une borne superieure, pas une variation reelle.
- **Le cout du raisonnement deplace l equilibre** : `A cout eleve, le cout du raisonnement peut inverser l equilibre observe.` alors que `a cout nul, (C,C) domine (D,D)`.
- **Le controle negatif discrimine** : `VERDICT CONTROLE NEGATIF : la borne serre a bien declenche SimulationTimeout.` — le cas serre rend `status=timeout`, le cas nominal rend `status=ok` : l instrument distingue budget suffisant et budget serre.


## Verifiez votre comprehension

1. Quelles bornes fixent le moteur de simulation, et que se passe-t-il quand le budget est epuise ?

   <details><summary>Reponse</summary>

   **Reponse.** Le moteur est configure par `Bornes : MAX_DEPTH=3, STEP_BUDGET=1000`. Quand le budget est epuise avant le verdict, il leve `Exception : SimulationTimeout levee avant verdict si budget epuise` : aucun verdict n est rendu dans ce cas. Une simulation incomplete est donc signalee comme telle, plutot que de renvoyer une issue calculee sous une borne cassee.

   </details>

2. Que rend le controle negatif du budget serre, et que permet-il de conclure sur l instrument ?

   <details><summary>Reponse</summary>

   **Reponse.** Avec un budget serre, la simulation s arrete en `status=timeout` et le duel se termine sur `a=D, b=D`, la ou le cas nominal rend `status=ok` avec `a=C, b=C`. L organe conclut `VERDICT CONTROLE NEGATIF : la borne serre a bien declenche SimulationTimeout.` : l instrument **discrimine** budget suffisant et budget serre. Un `timeout` signale donc un budget trop serre pour le probleme pose, pas un defaut du modele ni une reponse Negative du jeu.

   </details>


## Pour aller plus loin

**Carnet suivant.** [Dynamics-07 — Equilibres par simulation](Dynamics-07-Equilibres-par-Simulation-Python.ipynb) — quand le cout du raisonnement deplace l issue du Dilemme du prisonnier, la question suivante est de lire l equilibre lui-meme par simulation.

**Carnets voisins.**

- [Dynamics-05 — Agents bornes (Python)](Dynamics-05-Agents-Bornes-Python.ipynb) — la meme question du budget, prise du point de vue des agents : un `defectBot` a budget eleve change-t-il de comportement ?
- [Dynamics-08 — Programmes transparents comme institutions](Dynamics-08-Institutions-Transparentes-Python.ipynb) — la transparence des programmes, qui remplace la preuve par la lecture du code source de l adversaire.

**Lectures.**

- Critch, *A priori refinement*, arXiv 1602.04184 (2016) — la logique de refinement dont ce carnet herite la borne.
- Garrabrant, *Optimization Monitoring* (2018) — le monitoring d optimisation et la borne de cout.
- Papadimitriou & Tsitsiklis, *The complexity of optimal queueing network control* (1999) — le cout du calcul dans les theories de jeu.
- maisi.org, *Open-Source Game Theory* (page de direction de recherche) — l Open Problem 3 que l exercice 4 explore, et le Löb borné paramétrique comme programme.
- TorchLean, *week-08-bounded-lob* (repo `Robertboy18/TorchLean-Verified-Examples`, pin `5afa0589a4d9`) — le Löb borné paramétrique prouvé en Lean : la tête théorique du seuil que ce carnet mesure.


## References

- Critch, *A priori refinement*, arXiv 1602.04184 (2016).
- Garrabrant, *Optimization Monitoring* (2018).
- Papadimitriou & Tsitsiklis, *The complexity of optimal queueing network control*, 1999 (cout du calcul en theorie des jeux).
- Critch, Dennis & Russell, « Cooperative and uncooperative institution designs: Surprises and problems in open-source game theory », arXiv 2208.07006 (2022) — les théorèmes bornés dont l'exercice 4 explore la zone ouverte (Open Problem 3).

**État de l'art ouvert (comparaison du 2026-11-16, issue #26985).**

- maisi.org — *Open-Source Game Theory* (page de direction de recherche) : l'énoncé du Löb borné paramétrique (f(k) ≳ log k) et l'Open Problem 3 que l'exercice 4 explore.
- transhumanist_atom_understander — *Payorian cooperation is easy with Kripke frames*, Alignment Forum, 2026-03-09 (https://www.alignmentforum.org/posts/LaCP6WyNzX8kiZn3w/) : la coopération sans Löb, par preuves sémantiques sur cadres de Kripke — le contraste non borné de ce que ce carnet mesure borné.
- Duclaux, Formenti, Cobben, Zhang, Schölkopf & Jin (2026) — *open-source-game-theory* : mécanisation Lean complète (PA-S' sur lib Foundation, **Löb borné inconditionnel**, τ-bots, EGT des populations). Repo `ColombanD/open-source-game-theory`, pin `main@c7f0ed76773c` — sans licence : citation seule, aucune réutilisation de code.
- Math-for-AI-Safety — *MAIS, agendas* (A1 : Löb borné quantitatif — coût minimal F(k,n), seuils de coopération ; MAIS-O16 : budget de coopération minimal de FairBot). Repo `Math-for-AI-Safety/MAIS`, pin `1cb9f003d111`.
- Robertboy18 — *TorchLean-Verified-Examples*, week-07 / **week-08-bounded-lob** : le Löb borné paramétrique **prouvé** en Lean — ce que ce carnet mesure sans le prouver. Repo `Robertboy18/TorchLean-Verified-Examples`, pin `5afa0589a4d9`.
- Carnet precedent prolonge ici : [04 — Open-Source Game Theory](Dynamics-04-Open-Source-Game-Theory-Python.ipynb) (le moteur « 06e », instrument identique).
